# 14.3 Exchanging information with solvers via suffixes

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

We have seen that to represent values associated with a `model` component, AMPL
employs various qualifiers or suffixes appended to component names. A suffix consists
of a period or "dot" (.) followed by a (usually short) identifier, so that for example the
reduced cost associated with a variable Buy[j] is written Buy[j].rc, and the reduced
costs of all such variables can be viewed by giving the command `display` Buy.rc.
There are numerous built-in suffixes of this kind, summarized in the tables in A.11.
AMPL cannot anticipate all of the values that a solver might associate with `model`
components, however. The values recognized as input or computed as output depend on
the design of each solver and its algorithms. To provide for open-ended representation of
such values, new suffixes may be defined for the duration of an AMPL session, either by
the user for sending values to a solver, or by a solver for returning values.

This section introduces both user-defined and solver-defined suffixes, illustrated by
features of the CPLEX solver. We show how user-defined suffixes can pass preferences
for variable selection and branch direction to an `integer` programming solver. Sensitivity
analysis provides an example of solver-defined suffixes that have numeric values, while
infeasibility diagnosis shows how a symbolic (string-valued) suffix works. Reporting a
direction of unboundedness gives an example of a solver-defined suffix in an AMPL
script, where it must be declared before it can be used.

User-defined suffixes: `integer` programming directives
Most solvers recognize a variety of algorithmic choices or settings, each governed by
a single value that applies to the entire problem being solved. Thus you can alter selected
settings by setting up a single string of directives, as in this example applying the CPLEX
solver to an `integer` program:

```ampl
ampl: model multmip3.mod;
ampl: data multmip3.dat;
ampl: option solver cplex;
ampl: option cplex_options 'nodesel 3 varsel 1 backtrack 0.1';
ampl: solve;
CPLEX 8.0.0: nodesel 3
varsel 1
backtrack 0.1
CPLEX 8.0.0: optimal integer solution; objective 235625
1052 MIP simplex iterations
75 branch-and-bound nodes
```

A few kinds of solver settings are more complex, however, in that they require separate
values to be set for individual `model` components. These settings are far too numerous to
be accommodated in a directive string. Instead the solver interface can be set up to recognize
new suffixes that the user defines specially for the solver's purposes.

As an example, for each variable in an `integer` program, CPLEX recognizes a separate
branching priority and a separate preferred branching direction, represented by an `integer`
in [0, 9999] and in [-1, 1] respectively. AMPL's CPLEX driver recognizes the suffixes .priority
and .direction as giving these settings. To use these suffixes, we
begin by giving a suffix command to define each one for the current AMPL session:

```ampl
ampl: suffix priority IN, integer, >= 0, <= 9999;
ampl: suffix direction IN, integer, >= -1, <= 1;
```

The effect of these statements is to define expressions of the form name.priority and
name.direction, where name denotes any variable, `objective` or constraint of the current
`model`. The argument IN specifies that values corresponding to these suffixes are to
be read in by the solver, and the subsequent phrases place restrictions on the values that
will be accepted (much as in a `param` declaration).

The newly defined suffixes may be assigned values by the `let` command ([Section 11.3](../11/11_3_modifying_data.ipynb))
or by later declarations as described in Sections A.8, A.9, A.10, and A.18.8. For
our current example we want to use these suffixes to assign CPLEX priority and direction
values corresponding to the `binary` variables Use[i,j]. Normally such values are chosen
based on knowledge of the problem and past experience with similar problems. Here
is one possibility:

```ampl
ampl: let {i in ORIG, j in DEST}
ampl?    Use[i,j].priority := sum {p in PROD} demand[j,p];
ampl: let Use["GARY","FRE"].direction := -1;
```

Variables not assigned a .priority or .direction value get a default value of zero
(as do all constraints and objectives in this example), as you can `check`:

```ampl
ampl: display Use.direction;
Use.direction [*,*] (tr)
:   CLEV GARY PITT    :=
DET   0     0   0
FRA   0     0   0
FRE   0    -1   0
LAF   0     0   0
LAN   0     0   0
STL   0     0   0
WIN   0     0   0
;
```

With the suffix values assigned as shown, CPLEX's search for a solution turns out to
require fewer simplex iterations and fewer branch-and-bound nodes:

```ampl
ampl: option reset_initial_guesses 1;
ampl: solve;
CPLEX 8.0.0: nodesel 3
varsel 1
backtrack 0.1
CPLEX 8.0.0: optimal integer solution; objective 235625
799 MIP simplex iterations
69 branch-and-bound nodes
```

(We have set option reset_initial_guesses to 1 so that the optimal solution from
the first CPLEX run won't be sent back to the second.)
Further information about the suffixes recognized by CPLEX and how to determine
the corresponding settings can be found in the CPLEX driver documentation. Other
solver interfaces may recognize different suffixes for different purposes; you'll need to
`check` separately for each solver you want to use.
**Solver-defined suffixes: sensitivity analysis**

When the keyword sensitivity is included in CPLEX's list of directives, classical
sensitivity ranges are computed and are returned in three new suffixes, .up, .down, and
.current:

```ampl
ampl: model steelT.mod; data steelT.dat;
ampl: option solver cplex;
ampl: option cplex_options 'sensitivity';
ampl: solve;
CPLEX 8.0.0: sensitivity
CPLEX 8.0.0: optimal solution; objective 515033
16 dual simplex iterations (0 in phase I)
suffix up OUT;
suffix down OUT;
suffix current OUT;
```

The three lines at the end of the output from the `solve` command show the suffix
commands that are executed by AMPL in response to the results from the solver. These
statements are executed automatically; you do not need to type them. The argument OUT
in each command says that these are suffixes whose values will be written out by the
solver (in contrast to the previous example, where the argument IN indicated suffix values
that the solver was to read in).

The sensitivity suffixes are interpreted as follows. For variables, suffix .current
indicates the `objective` function coefficient in the current problem, while .down and .up
give the smallest and largest values of the `objective` coefficient for which the current LP
basis remains optimal:

```ampl
ampl:    display Sell.down, Sell.current, Sell.up;
:          Sell.down Sell.current    Sell.up       :=
bands    1   23.3          25       1e+20
bands    2   25.4          26       1e+20
bands    3   24.9          27          27.5
bands    4   10            27          29.1
coils    1   29.2857       30          30.8571
coils    2   33            35       1e+20
coils    3   35.2857       37       1e+20
coils    4   35.2857       39       1e+20
;
```

For constraints, the interpretation is similar except that it applies to a constraint's constant
term (the so-called right-hand-side value):

```ampl
ampl: display Time.down, Time.current, Time.up;
: Time.down Time.current   Time.up    :=
1   37.8071       40       66.3786
2   37.8071       40       47.8571
3   25            32       45
4   30            40       62.5
;
```

You can use generic synonyms ([Section 12.6](../12/12_6_accessing_model_and_solver_status.ipynb)) to `display` a `table` of ranges for all variables
or constraints, similar to the tables produced by the standalone version of CPLEX. (Values
of -1e+20 in the .down column and 1e+20 in the .up column correspond to what
CPLEX calls -infinity and +infinity in its tables.)

#### Solver-defined suffixes: infeasibility diagnosis

For a linear program that has no feasible solution, you can ask CPLEX to find an irreducible
infeasible subset (or IIS): a collection of constraints and variable bounds that is
infeasible but that becomes feasible when any one constraint or bound is removed. If a
small IIS exists and can be found, it can provide valuable clues as to the source of the
infeasibility. You turn on the IIS finder by changing the iisfind directive from its
default value of 0 to either 1 (for a relatively fast version) or 2 (for a slower version that
tends to find a smaller IIS).

The following example shows how IIS finding might be applied to the infeasible diet
problem from [Chapter 2](../02/02.md). After `solve` detects that there is no feasible solution, it is
repeated with the directive 'iisfind 1':

```ampl
ampl: model diet.mod; data diet2.dat; option solver cplex;
ampl: solve;
CPLEX 8.0.0: infeasible problem.

4 dual simplex iterations (0 in phase I)
constraint.dunbdd returned
suffix dunbdd OUT;

ampl: option cplex_options 'iisfind 1';
ampl: solve;
CPLEX 8.0.0: iisfind 1
CPLEX 8.0.0: infeasible problem.

0 simplex iterations (0 in phase I)
Returning iis of 7 variables and 2 constraints.
constraint.dunbdd returned

suffix iis symbolic OUT;
option iis_table '\
0       non     not in the iis\
1       low     at lower bound\
2       fix     fixed\
3       upp     at upper bound\
';
```

Again, AMPL shows any suffix statement that has been executed automatically. Our
interest is in the new suffix named .iis, which is symbolic, or string-valued. An
associated option iis_table, also set up by the solver driver and displayed automatically
by `solve`, shows the strings that may be associated with .iis and gives brief
descriptions of what they mean.

You can use `display` to look at the .iis values that have been returned:

```ampl
ampl: display _varname, _var.iis, _conname, _con.iis;
:     _varname    _var.iis     _conname    _con.iis :=
1   "Buy['BEEF']"   upp      "Diet['A']"     non
2   "Buy['CHK']"    low      "Diet['B1']"    non
3   "Buy['FISH']"   low      "Diet['B2']"    low
4   "Buy['HAM']"    upp      "Diet['C']"     non
5   "Buy['MCH']"    non      "Diet['NA']"    upp
6   "Buy['MTL']"    upp      "Diet['CAL']"   non
7   "Buy['SPG']"    low            .             .
8   "Buy['TUR']"    low            .             .
;
```

This information indicates that the IIS consists of four lower and three upper bounds on
the variables, plus the constraints providing the lower bound on B2 and the upper bound
on NA in the diet. Together these restrictions have no feasible solution, but dropping any
one of them will permit a solution to be found to the remaining ones.

If dropping the bounds is not of interest, then you may want to list only the constraints
in the IIS. A print statement produces a concise listing:

```ampl
ampl: print {i in 1.._ncons:
ampl?    _con[i].iis <> "non"}: _conname[i];
Diet['B2']
Diet['NA']
```

You could conclude in this case that, to avoid violating the bounds on amounts purchased,
you might need to accept either less vitamin B2 or more sodium, or both, in the
diet. Further experimentation would be necessary to determine how much less or more,
however, and what other changes you might need to accept in order to gain feasibility.
(A linear program can have several irreducible infeasible subsets, but CPLEX's IIS-
findingalgorithm detects only one IIS at a time.)

**Solver-defined suffixes: direction of unboundedness**

For an unbounded linear program — one that has in effect a minimum of -Infinity
or a maximum of +Infinity — a solver can return a ray of feasible solutions
of the form X + α d, where α ≥ 0. On return from CPLEX, the feasible solution X
is given by the values of the variables, while the direction of unboundedness d is given by
an additional value associated with each variable through the solver-defined suffix
.unbdd.

An application of the direction of unboundedness can be found in a `model`
trnloc1d.mod and script trnloc1d.run for Benders decomposition applied to a
combination of a warehouse-location and transportation problem; the `model`, `data` and
script are available from the AMPL web site. We won't try to describe the whole decomposition
scheme here, but rather concentrate on the subproblem obtained by fixing the
zero-one variables Build[i], which indicate the warehouses that are to be built, to trial
values build[i]. In its dual form, this subproblem is:

```ampl
var Supply_Price {ORIG} <= 0;
var Demand_Price {DEST};
maximize Dual_Ship_Cost:
       sum {i in ORIG} Supply_Price[i] * supply[i] * build[i] +
       sum {j in DEST} Demand_Price[j] * demand[j];
subject to Dual_Ship {i in ORIG, j in DEST}:
       Supply_Price[i] + Demand_Price[j] <= var_cost[i,j];
```

When all values build[i] are set to zero, no warehouses are built, and the primal subproblem
is infeasible. As a result, the dual formulation of the subproblem, which always
has a feasible solution, must be unbounded.

As the remainder of this chapter will explain, we `solve` a subproblem by collecting its
components into an AMPL "problem" and then directing AMPL to `solve` only that problem.
When this approach is applied to the dual subproblem from the AMPL commandline,
CPLEX returns the direction of unboundedness in the expected way:

```ampl
ampl: model trnloc1d.mod;
ampl: data trnloc1.dat;
ampl: problem Sub: Supply_Price, Demand_Price,
ampl?    Dual_Ship_Cost, Dual_Ship;
ampl: let {i in ORIG} build[i] := 0;
ampl: option solver cplex, cplex_options 'presolve 0';
ampl: solve;
CPLEX 8.0.0: presolve 0
CPLEX 8.0.0: unbounded problem.

25 dual simplex iterations (25 in phase I)
variable.unbdd returned
6 extra simplex iterations for ray (1 in phase I)
suffix unbdd OUT;
```

The suffix message indicates that .unbdd has been created automatically. You can
use this suffix to `display` the direction of unboundedness, which is simple in this case:

```ampl
ampl: display Supply_Price.unbdd;
Supply_Price.unbdd [*] :=
       1 -1  4 -1   7 -1 10 -1 13 -1  16 -1   19 -1   22 -1   25 -1
       2 -1  5 -1   8 -1 11 -1 14 -1  17 -1   20 -1   23 -1
       3 -1  6 -1   9 -1 12 -1 15 -1  18 -1   21 -1   24 -1
;
ampl: display Demand_Price.unbdd;
Demand_Price.unbdd [*] :=
A3 1
A6 1
A8 1
A9 1
B2 1
B4 1
;
```

Our script for Benders decomposition (trnloc1d.run) solves the subproblem repeatedly,
with differing build[i] values generated from the master problem. After each
`solve`, the result is tested for unboundedness and an extension of the master problem is
constructed accordingly. The essentials of the main loop are as follows:

```ampl
repeat {
       solve Sub;
       if Dual_Ship_Cost <= Max_Ship_Cost + 0.00001 then break;
       if Sub.result = "unbounded" then {
       let nCUT := nCUT + 1;
       let cut_type[nCUT] := "ray";
       let {i in ORIG}
              supply_price[i,nCUT] := Supply_Price[i].unbdd;
       let {j in DEST}
              demand_price[j,nCUT] := Demand_Price[j].unbdd;
       } else {
       let nCUT := nCUT + 1;
       let cut_type[nCUT] := "point";
       let {i in ORIG} supply_price[i,nCUT] := Supply_Price[i];
       let {j in DEST} demand_price[j,nCUT] := Demand_Price[j];
       }
       solve Master;
       let {i in ORIG} build[i] := Build[i];
}
```

An attempt to use .unbdd in this context fails, however:

```ampl
ampl: commands trnloc1d.run;
trnloc1d.run, line 39 (offset 931):
              Bad suffix .unbdd for Supply_Price
context: let {i in ORIG} supply_price[i,nCUT] :=
                     >>> Supply_Price[i].unbdd; <<<
```

The difficulty here is that AMPL scans all commands in the repeat loop before
beginning to execute any of them. As a result it encounters the use of .unbdd before
any infeasible subproblem has had a chance to cause this suffix to be defined. To make
this script run as intended, it is necessary to place the statement

```ampl
suffix unbdd OUT;
```

in the script before the repeat loop, so that .unbdd is already defined at the time the
loop is scanned.

Defining and using suffixes
A new AMPL suffix is defined by a statement consisting of the keyword suffix followed
by a suffix-name and then one or more optional qualifiers that indicate what values
may be associated with the suffix and how it may be used. For example, we have seen
the definition

```ampl
suffix priority IN, integer, >= 0, <= 9999;
```

for the suffix priority with in-out, type, and bound qualifiers.
The suffix statement causes AMPL to recognize suffixed expressions of the form
component-name.suffix-name, where component-name refers to any currently declared
variable, constraint, or `objective` (or problem, as defined in the next section). The definition
of a suffix remains in effect until the next `reset` command or the end of the current
AMPL session. The suffix-name is `subject to` the same rules as other names in AMPL.
Suffixes have a separate name space, however, so a suffix may have the same name as a
parameter, variable, or other `model` component. The optional qualifiers of the suffix
statement may appear in any order; their forms and effects are described below.

The optional type qualifier in a suffix statement indicates what values may be associated
with the suffixed expressions, with all numeric values being the default:

```ampl
suffix type      values allowed
none specified   any numeric value
integer          integer numeric values
binary           0 or 1
symbolic         character strings listed in option suffix-name_table
```

All numeric-valued suffixed expressions have an initial value of 0. Their permissible values
may be further limited by one or two bound qualifiers of the form

```ampl
>= arith-expr
<= arith-expr
```

where arith-expr is any arithmetic expression not involving variables.

For each symbolic suffix, AMPL automatically defines an associated numeric suffix
, suffix-name_num. An AMPL option suffix-name_table must then be created to
define a relation between the .suffix-name and .suffix-name_num values, as in the following
example:

```ampl
suffix iis symbolic OUT;
option iis_table '\
0       non     not in the iis\
1       low     at lower bound\
2       fix     fixed\
3       upp     at upper bound\
';
```

Each line of the `table` consists of an `integer` value, a string value, and an optional comment.
Every string value is associated with its adjacent `integer` value, and with any
higher `integer` values that are less than the `integer` on the next line. Assigning a string
value to a .suffix-name expression is equivalent to assigning the associated numeric
value to a .suffix-name_num expression. The latter expressions are initially assigned the
value 0, and are `subject to` any type and bound qualifiers as described above. (Normally
the string values of symbolic suffixes are used in AMPL commands and scripts, while
the numeric values are used in communication with solvers.)
The optional in-out qualifier determines how suffix values interact with the solver:

```ampl
in-out    handling of suffix values
IN        written by AMPL before invoking the solver, then read in by solver
OUT       written out by solver, then read by AMPL after the solver is finished
INOUT     both read and written, as for IN and OUT above
LOCAL     neither read nor written
```

INOUT is the default if no in-out keyword is specified.

We have seen that suffixed expressions can be assigned or reassigned values by a `let`
statement:

```ampl
let Use["GARY","FRE"].direction := -1;
```

Here just one variable is assigned a suffixed value, but often there are suffixed values for
all variables in an indexed collection:

```ampl
var Use {ORIG,DEST} binary;
let {i in ORIG, j in DEST}
       Use[i,j].priority := sum {p in PROD} demand[j,p];
```

In this case the assignment of suffix values can be combined with the variable's declaration
:

```ampl
var Use {i in ORIG, j in DEST} binary,
       suffix priority sum {p in PROD} demand[j,p];
```

In general, one or more of the phrases in a `var` declaration may consist of the keyword
suffix followed by a previously-defined suffix-name and an expression for evaluating
the associated suffix expressions.